In [0]:
import requests
import pandas as pd
from pyspark.sql.functions import current_timestamp

# BCB SGS series configuration
# 11 = Daily Selic interest rate
series_code = 11
series_name = "selic_daily"

# Date range
start_date = "01/01/2023"
end_date = "31/12/2025"

# BCB API URL
url = (
    f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{series_code}/dados"
    f"?formato=json&dataInicial={start_date}&dataFinal={end_date}"
)

response = requests.get(url)

if response.status_code != 200:
    raise Exception(f"API request failed: {response.status_code} - {response.text}")

data = response.json()

# Convert JSON response to Pandas DataFrame
df = pd.DataFrame(data)

# Convert columns to proper data types
df["data"] = pd.to_datetime(df["data"], format="%d/%m/%Y")
df["valor"] = df["valor"].str.replace(",", ".", regex=False).astype(float)

# Add metadata columns
df["series_code"] = series_code
df["series_name"] = series_name

# Convert Pandas DataFrame to Spark DataFrame
spark_df = spark.createDataFrame(df)

# Create catalog and schema
spark.sql("CREATE CATALOG IF NOT EXISTS portfolio")
spark.sql("CREATE SCHEMA IF NOT EXISTS portfolio.bcb_economic_indicators")

# Add ingestion timestamp
spark_df = spark_df.withColumn("ingestion_timestamp", current_timestamp())

# Save as Delta table
spark_df.write.mode("overwrite").saveAsTable(
    "portfolio.bcb_economic_indicators.raw_selic_daily"
)

display(spark_df)